# EP21. 코드 리뷰 에이전트

## 시니어 개발자 리뷰 시간을 70% 줄인 방법

---

### 학습 목표

1. **PR diff 파싱**: Unified diff format을 파일별로 분리하고 변경사항을 구조화하는 방법을 익힌다
2. **전문 분석 에이전트**: 보안 취약점, 코드 스멜, 컨벤션 체크를 수행하는 3개의 전문 에이전트를 구현한다
3. **멀티에이전트 파이프라인**: LangGraph로 3개 에이전트를 병렬 실행하는 통합 리뷰 시스템을 구축한다
4. **품질 추적**: 파일별 모델 라우팅으로 비용을 최적화하고, Langfuse로 리뷰 품질을 추적한다

---

### AI 가이드

이 노트북은 GitHub API 없이 **Mock PR diff 데이터**로 동작합니다.  
실제 GitHub 연동은 EP12(MCP Server)에서 다룬 GitHub MCP를 활용하여 확장할 수 있습니다.

### 사전 요구사항

- **EP10 멀티에이전트**: LangGraph StateGraph, 병렬 실행 개념
- **EP12 MCP**: MCP 서버 기초 (GitHub 연동 확장 시)
- **EP16 LLM FinOps**: 모델 라우팅 전략

| 항목 | 내용 |
|------|------|
| 소요 시간 | 약 80분 |
| 난이도 | ⭐⭐⭐ |
| API 키 | `ANTHROPIC_API_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` (선택) |

---

## Section 1: 환경 설정

In [ ]:
# 필수 패키지 설치
!uv pip install fastmcp langgraph langchain langchain-anthropic langfuse pydantic python-dotenv --quiet

---

## Section 2: 라이브러리 로드 + API 키 설정

In [ ]:
import os
import json
import re
from typing import TypedDict, Annotated, Literal
from operator import add
from dataclasses import dataclass, field

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

# API 키 확인
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요."
print("Anthropic API 키 설정 완료")

# Langfuse (선택 - 없으면 건너뜀)
LANGFUSE_ENABLED = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))
if LANGFUSE_ENABLED:
    from langfuse import Langfuse
    from langfuse.langchain import CallbackHandler as LangfuseCallbackHandler
    langfuse_client = Langfuse()
    print("Langfuse 연동 활성화")
else:
    print("Langfuse 키 미설정 - 품질 추적 섹션은 건너뜁니다")

---

## Section 3: 샘플 PR Diff 데이터

실제 GitHub PR 없이 동작하도록 **의도적인 문제가 포함된** Mock diff를 사용합니다.  
포함된 문제: SQL Injection, 긴 함수, 잘못된 네이밍, 하드코딩된 비밀번호 등

In [ ]:
SAMPLE_PR_DIFF = r"""
diff --git a/app/auth.py b/app/auth.py
index abc1234..def5678 100644
--- a/app/auth.py
+++ b/app/auth.py
@@ -1,8 +1,45 @@
 import os
 import sqlite3
+import hashlib
 from flask import Flask, request, jsonify
 
 app = Flask(__name__)
+DB_PASSWORD = "admin123!@#"
+api_key = "sk-proj-abc123def456ghi789"
 
+def getUserData(username, db_path):
+    conn = sqlite3.connect(db_path)
+    cursor = conn.cursor()
+    query = f"SELECT * FROM users WHERE username = '{username}'"
+    cursor.execute(query)
+    result = cursor.fetchall()
+    if result:
+        for row in result:
+            if row[2] == 'admin':
+                if row[3] > 0:
+                    if row[4] != None:
+                        data = {}
+                        data['name'] = row[0]
+                        data['email'] = row[1]
+                        data['role'] = row[2]
+                        data['score'] = row[3]
+                        data['extra'] = row[4]
+                        data['status'] = 'active'
+                        data['timestamp'] = os.popen('date').read()
+                        return data
+    return None
+
+@app.route('/login', methods=['POST'])
+def login():
+    u = request.form.get('username')
+    p = request.form.get('password')
+    q = f"SELECT * FROM users WHERE username='{u}' AND password='{p}'"
+    conn = sqlite3.connect('users.db')
+    r = conn.cursor().execute(q).fetchone()
+    if r:
+        return jsonify({'status': 'ok', 'token': hashlib.md5(u.encode()).hexdigest()})
+    return jsonify({'status': 'fail'}), 401
+
+@app.route('/search')
+def search():
+    keyword = request.args.get('q', '')
+    return f"<h1>Results for: {keyword}</h1>"
 
 if __name__ == '__main__':
+    app.run(debug=True)

diff --git a/app/utils.py b/app/utils.py
new file mode 100644
index 0000000..1234567
--- /dev/null
+++ b/app/utils.py
@@ -0,0 +1,18 @@
+import os
+
+def calc(x):
+    # x를 계산한다
+    result = x * 2
+    return result
+
+def processData(items):
+    Res = []
+    for i in items:
+        val = i * 3.14159
+        if val > 100:
+            Res.append(val)
+    return Res
+
+def do_something(a, b, c, d, e, f, g):
+    return a + b + c + d + e + f + g
+
"""

print(f"샘플 PR diff 준비 완료 ({len(SAMPLE_PR_DIFF)} characters)")
print("포함된 의도적 문제:")
print("  - SQL Injection (f-string 쿼리)")
print("  - XSS (사용자 입력 직접 HTML 출력)")
print("  - Secrets Exposure (하드코딩된 비밀번호, API 키)")
print("  - 코드 스멜 (깊은 중첩, 매직 넘버, 긴 파라미터 목록)")
print("  - 컨벤션 위반 (camelCase 함수명, 대문자 변수, 타입 힌트 누락)")

---

## Section 4: PR Diff 파서 구현

Unified diff format을 파일 단위로 분리하고 구조화합니다.

In [ ]:
@dataclass
class FileDiff:
    """파일 단위 diff 정보"""
    file_path: str
    change_type: str  # 'modified', 'added', 'deleted'
    diff_content: str
    added_lines: int = 0
    deleted_lines: int = 0
    added_code: list = field(default_factory=list)


def parse_pr_diff(raw_diff: str) -> list[FileDiff]:
    """Unified diff를 파일별 FileDiff 객체로 파싱한다.
    
    Args:
        raw_diff: GitHub PR에서 가져온 unified diff 문자열
    
    Returns:
        파일별로 분리된 FileDiff 객체 리스트
    """
    file_diffs = []
    
    # 'diff --git' 기준으로 파일별 분리
    file_sections = re.split(r'(?=^diff --git)', raw_diff.strip(), flags=re.MULTILINE)
    file_sections = [s for s in file_sections if s.strip()]
    
    for section in file_sections:
        # 파일 경로 추출
        path_match = re.search(r'diff --git a/(.*?) b/(.*)', section)
        if not path_match:
            continue
        
        file_path = path_match.group(2)
        
        # 변경 유형 결정
        if 'new file mode' in section:
            change_type = 'added'
        elif 'deleted file mode' in section:
            change_type = 'deleted'
        else:
            change_type = 'modified'
        
        # 추가/삭제 라인 수 계산
        lines = section.split('\n')
        added_lines = sum(1 for l in lines if l.startswith('+') and not l.startswith('+++'))
        deleted_lines = sum(1 for l in lines if l.startswith('-') and not l.startswith('---'))
        
        # 추가된 코드만 추출 (리뷰 대상)
        added_code = [l[1:] for l in lines if l.startswith('+') and not l.startswith('+++')]
        
        file_diffs.append(FileDiff(
            file_path=file_path,
            change_type=change_type,
            diff_content=section,
            added_lines=added_lines,
            deleted_lines=deleted_lines,
            added_code=added_code,
        ))
    
    return file_diffs

In [ ]:
# 파서 실행 및 결과 확인
file_diffs = parse_pr_diff(SAMPLE_PR_DIFF)

print(f"총 {len(file_diffs)}개 파일 변경 감지\n")

for fd in file_diffs:
    print(f"{'='*60}")
    print(f"파일: {fd.file_path}")
    print(f"변경 유형: {fd.change_type}")
    print(f"추가: +{fd.added_lines} / 삭제: -{fd.deleted_lines}")
    print(f"추가된 코드 (처음 5줄):")
    for line in fd.added_code[:5]:
        print(f"  + {line}")
    print()

---

## Section 5: 보안 취약점 분석 에이전트

OWASP Top 10 기준으로 보안 이슈를 탐지하는 에이전트입니다.  
Structured Output(Pydantic)으로 결과를 안정적으로 구조화합니다.

In [ ]:
# --- Pydantic 스키마 정의 ---

class SecurityIssue(BaseModel):
    """보안 취약점 이슈"""
    severity: Literal["CRITICAL", "HIGH", "MEDIUM", "LOW"] = Field(
        description="취약점 심각도"
    )
    category: str = Field(
        description="OWASP 카테고리 (예: SQL_INJECTION, XSS, SECRETS_EXPOSURE)"
    )
    line: str = Field(
        description="문제가 되는 코드 라인"
    )
    description: str = Field(
        description="취약점에 대한 한국어 설명"
    )
    suggestion: str = Field(
        description="구체적인 수정 방법"
    )


class SecurityAnalysis(BaseModel):
    """보안 분석 결과"""
    issues: list[SecurityIssue] = Field(default_factory=list)
    summary: str = Field(description="전체 보안 상태 요약 (한국어)")


# --- 시스템 프롬프트 ---

SECURITY_SYSTEM_PROMPT = """당신은 시니어 보안 엔지니어입니다.
PR diff에서 추가된 코드(+ 로 시작하는 라인)를 분석하여 OWASP Top 10 관점에서 보안 취약점을 찾아주세요.

주요 탐지 항목:
- SQL Injection: f-string, format, % 연산으로 구성된 SQL 쿼리
- XSS: 사용자 입력을 이스케이프 없이 HTML에 직접 출력
- Secrets Exposure: API 키, 비밀번호, 토큰 등의 하드코딩
- Command Injection: os.system, subprocess + 사용자 입력 결합
- Insecure Hashing: MD5, SHA1 등 약한 해시 알고리즘 사용
- Path Traversal: 사용자 입력을 파일 경로에 직접 사용

각 이슈에 대해 severity, category, line, description(한국어), suggestion을 제공하세요.
이슈가 없으면 빈 리스트를 반환하세요."""

print("보안 에이전트 스키마 및 프롬프트 정의 완료")

In [ ]:
def run_security_agent(diff_content: str, model_name: str = "claude-sonnet-4-20250514") -> SecurityAnalysis:
    """보안 취약점 분석 에이전트를 실행한다.
    
    Args:
        diff_content: 분석할 diff 문자열
        model_name: 사용할 Claude 모델명
    
    Returns:
        SecurityAnalysis: 구조화된 보안 분석 결과
    """
    llm = ChatAnthropic(model=model_name, temperature=0)
    structured_llm = llm.with_structured_output(SecurityAnalysis)
    
    result = structured_llm.invoke([
        SystemMessage(content=SECURITY_SYSTEM_PROMPT),
        HumanMessage(content=f"다음 PR diff를 보안 관점에서 분석하세요:\n\n{diff_content}"),
    ])
    
    return result


# 보안 에이전트 실행 (auth.py)
auth_diff = file_diffs[0]  # app/auth.py
print(f"분석 대상: {auth_diff.file_path} (+{auth_diff.added_lines} lines)\n")

security_result = run_security_agent(auth_diff.diff_content)

print(f"=== 보안 분석 결과 ===")
print(f"발견된 이슈: {len(security_result.issues)}개\n")

for i, issue in enumerate(security_result.issues, 1):
    print(f"--- 이슈 {i} ---")
    print(f"  심각도: {issue.severity}")
    print(f"  카테고리: {issue.category}")
    print(f"  코드: {issue.line}")
    print(f"  설명: {issue.description}")
    print(f"  수정 제안: {issue.suggestion}")
    print()

print(f"\n요약: {security_result.summary}")

---

## Section 6: 코드 스멜 탐지 에이전트

클린 코드 원칙에 따라 코드 스멜을 탐지하는 에이전트입니다.

In [ ]:
class SmellIssue(BaseModel):
    """코드 스멜 이슈"""
    type: str = Field(
        description="스멜 유형 (long_method, duplicated_code, complex_condition, magic_number, too_many_params, dead_code)"
    )
    severity: Literal["HIGH", "MEDIUM", "LOW"] = Field(
        description="심각도"
    )
    location: str = Field(
        description="해당 코드 위치 (함수명 또는 라인)"
    )
    description: str = Field(
        description="문제 설명 (한국어)"
    )
    refactoring: str = Field(
        description="구체적 리팩토링 방법"
    )


class SmellAnalysis(BaseModel):
    """코드 스멜 분석 결과"""
    issues: list[SmellIssue] = Field(default_factory=list)
    summary: str = Field(description="전체 코드 품질 요약 (한국어)")


SMELL_SYSTEM_PROMPT = """당신은 클린 코드 전문가입니다.
PR diff에서 추가된 코드(+ 로 시작하는 라인)를 분석하여 코드 스멜을 탐지해주세요.

주요 탐지 항목:
- Long Method: 함수가 과도하게 길거나 여러 책임을 가짐
- Duplicated Code: 유사한 코드 패턴이 반복됨
- Complex Conditions: 깊은 중첩 if문, 복잡한 조건식
- Magic Numbers: 의미 없는 숫자 리터럴 (상수 미정의)
- Too Many Parameters: 함수 파라미터가 과도하게 많음 (4개 초과)
- Dead Code: 도달 불가능하거나 사용되지 않는 코드
- God Class: 과도하게 많은 책임을 가진 클래스

각 이슈에 대해 type, severity, location, description(한국어), refactoring을 제공하세요."""

print("코드 스멜 에이전트 스키마 및 프롬프트 정의 완료")

In [ ]:
def run_smell_agent(diff_content: str, model_name: str = "claude-sonnet-4-20250514") -> SmellAnalysis:
    """코드 스멜 탐지 에이전트를 실행한다."""
    llm = ChatAnthropic(model=model_name, temperature=0)
    structured_llm = llm.with_structured_output(SmellAnalysis)
    
    result = structured_llm.invoke([
        SystemMessage(content=SMELL_SYSTEM_PROMPT),
        HumanMessage(content=f"다음 PR diff에서 코드 스멜을 탐지하세요:\n\n{diff_content}"),
    ])
    
    return result


# 코드 스멜 에이전트 실행 (auth.py + utils.py 통합)
full_diff = SAMPLE_PR_DIFF
print("분석 대상: 전체 PR diff\n")

smell_result = run_smell_agent(full_diff)

print(f"=== 코드 스멜 분석 결과 ===")
print(f"발견된 이슈: {len(smell_result.issues)}개\n")

for i, issue in enumerate(smell_result.issues, 1):
    print(f"--- 이슈 {i} ---")
    print(f"  유형: {issue.type}")
    print(f"  심각도: {issue.severity}")
    print(f"  위치: {issue.location}")
    print(f"  설명: {issue.description}")
    print(f"  리팩토링: {issue.refactoring}")
    print()

print(f"\n요약: {smell_result.summary}")

---

## Section 7: 컨벤션 체크 에이전트

PEP 8 및 팀 컨벤션 기준으로 스타일 위반을 탐지하는 에이전트입니다.

In [ ]:
class ConventionIssue(BaseModel):
    """컨벤션 위반 이슈"""
    rule: str = Field(
        description="위반된 규칙 (naming, docstring, import_order, type_hint, line_length, unnecessary_comment)"
    )
    severity: Literal["HIGH", "MEDIUM", "LOW"] = Field(
        description="심각도"
    )
    line: str = Field(
        description="해당 코드 라인"
    )
    fix: str = Field(
        description="수정된 코드 제안"
    )


class ConventionAnalysis(BaseModel):
    """컨벤션 분석 결과"""
    issues: list[ConventionIssue] = Field(default_factory=list)
    summary: str = Field(description="전체 컨벤션 상태 요약 (한국어)")


CONVENTION_SYSTEM_PROMPT = """당신은 Python 코드 스타일 리뷰어입니다.
PEP 8 및 다음 팀 컨벤션을 기준으로 PR diff를 분석해주세요.

검사 항목:
- naming: 함수/변수는 snake_case, 클래스는 PascalCase
- docstring: 공개 함수에는 docstring 필수
- import_order: stdlib -> 3rd party -> local 순서
- type_hint: 공개 함수의 파라미터와 반환값에 타입 힌트 권장
- line_length: 최대 88자 (Black 기준)
- unnecessary_comment: 코드만으로 명확한 내용의 불필요한 주석

추가된 코드(+ 로 시작하는 라인)만 분석하세요.
각 이슈에 대해 rule, severity, line, fix를 제공하세요."""

print("컨벤션 체크 에이전트 스키마 및 프롬프트 정의 완료")

In [ ]:
def run_convention_agent(diff_content: str, model_name: str = "claude-sonnet-4-20250514") -> ConventionAnalysis:
    """컨벤션 체크 에이전트를 실행한다."""
    llm = ChatAnthropic(model=model_name, temperature=0)
    structured_llm = llm.with_structured_output(ConventionAnalysis)
    
    result = structured_llm.invoke([
        SystemMessage(content=CONVENTION_SYSTEM_PROMPT),
        HumanMessage(content=f"다음 PR diff의 컨벤션 위반을 체크하세요:\n\n{diff_content}"),
    ])
    
    return result


# 컨벤션 에이전트 실행
print("분석 대상: 전체 PR diff\n")

convention_result = run_convention_agent(full_diff)

print(f"=== 컨벤션 분석 결과 ===")
print(f"발견된 이슈: {len(convention_result.issues)}개\n")

for i, issue in enumerate(convention_result.issues, 1):
    print(f"--- 이슈 {i} ---")
    print(f"  규칙: {issue.rule}")
    print(f"  심각도: {issue.severity}")
    print(f"  코드: {issue.line}")
    print(f"  수정: {issue.fix}")
    print()

print(f"\n요약: {convention_result.summary}")

---

## Section 8: 멀티에이전트 통합 파이프라인 (LangGraph)

3개 에이전트를 **병렬로 실행**하고 결과를 통합하는 LangGraph 파이프라인입니다.  
EP10에서 배운 `StateGraph` + `Send` API를 활용합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END

# --- State 정의 ---

class ReviewState(TypedDict):
    """코드 리뷰 파이프라인의 공유 상태"""
    pr_diff: str                                          # 원본 PR diff
    file_diffs: list                                      # 파싱된 FileDiff 리스트
    security_issues: Annotated[list, add]                 # 보안 이슈 (누적)
    smell_issues: Annotated[list, add]                    # 코드 스멜 (누적)
    convention_issues: Annotated[list, add]               # 컨벤션 위반 (누적)
    review_report: str                                    # 최종 리뷰 리포트

print("ReviewState 정의 완료")
print("  - security_issues, smell_issues, convention_issues에 Annotated[list, add] 사용")
print("  - 병렬 실행된 에이전트 결과가 자동으로 리스트에 누적됩니다")

In [ ]:
# --- 노드 함수 정의 ---

def parse_diff_node(state: ReviewState) -> dict:
    """PR diff를 파일별로 파싱한다."""
    file_diffs = parse_pr_diff(state["pr_diff"])
    print(f"[parse_diff] {len(file_diffs)}개 파일 파싱 완료")
    return {"file_diffs": file_diffs}


def security_agent_node(state: ReviewState) -> dict:
    """보안 취약점 분석 에이전트 노드."""
    all_issues = []
    for fd in state["file_diffs"]:
        model = select_model_for_file(fd)
        result = run_security_agent(fd.diff_content, model_name=model)
        all_issues.extend([issue.model_dump() for issue in result.issues])
    print(f"[security_agent] {len(all_issues)}개 보안 이슈 발견")
    return {"security_issues": all_issues}


def smell_agent_node(state: ReviewState) -> dict:
    """코드 스멜 탐지 에이전트 노드."""
    combined_diff = "\n".join(fd.diff_content for fd in state["file_diffs"])
    model = select_model_for_file(state["file_diffs"][0]) if state["file_diffs"] else "claude-sonnet-4-20250514"
    result = run_smell_agent(combined_diff, model_name=model)
    issues = [issue.model_dump() for issue in result.issues]
    print(f"[smell_agent] {len(issues)}개 코드 스멜 발견")
    return {"smell_issues": issues}


def convention_agent_node(state: ReviewState) -> dict:
    """컨벤션 체크 에이전트 노드."""
    combined_diff = "\n".join(fd.diff_content for fd in state["file_diffs"])
    model = select_model_for_file(state["file_diffs"][0]) if state["file_diffs"] else "claude-sonnet-4-20250514"
    result = run_convention_agent(combined_diff, model_name=model)
    issues = [issue.model_dump() for issue in result.issues]
    print(f"[convention_agent] {len(issues)}개 컨벤션 위반 발견")
    return {"convention_issues": issues}


def merge_results_node(state: ReviewState) -> dict:
    """3개 에이전트 결과를 통합하고 우선순위로 정렬한다."""
    total_issues = (
        len(state.get("security_issues", []))
        + len(state.get("smell_issues", []))
        + len(state.get("convention_issues", []))
    )
    
    severity_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
    
    # 보안 이슈를 심각도순으로 정렬
    sec = sorted(
        state.get("security_issues", []),
        key=lambda x: severity_order.get(x.get("severity", "LOW"), 4)
    )
    
    report_parts = []
    report_parts.append(f"# 코드 리뷰 리포트")
    report_parts.append(f"\n총 {total_issues}개 이슈 발견\n")
    
    report_parts.append(f"## 보안 취약점 ({len(sec)}개)")
    for issue in sec:
        report_parts.append(f"- [{issue.get('severity')}] {issue.get('category')}: {issue.get('description')}")
    
    smell = state.get("smell_issues", [])
    report_parts.append(f"\n## 코드 스멜 ({len(smell)}개)")
    for issue in smell:
        report_parts.append(f"- [{issue.get('severity')}] {issue.get('type')}: {issue.get('description')}")
    
    conv = state.get("convention_issues", [])
    report_parts.append(f"\n## 컨벤션 위반 ({len(conv)}개)")
    for issue in conv:
        report_parts.append(f"- [{issue.get('severity')}] {issue.get('rule')}: {issue.get('line')}")
    
    report = "\n".join(report_parts)
    print(f"[merge_results] 총 {total_issues}개 이슈 통합 완료")
    return {"review_report": report}


print("노드 함수 정의 완료: parse_diff, security_agent, smell_agent, convention_agent, merge_results")

In [ ]:
# --- LangGraph 파이프라인 구성 ---

builder = StateGraph(ReviewState)

# 노드 등록
builder.add_node("parse_diff", parse_diff_node)
builder.add_node("security_agent", security_agent_node)
builder.add_node("smell_agent", smell_agent_node)
builder.add_node("convention_agent", convention_agent_node)
builder.add_node("merge_results", merge_results_node)

# 엣지 연결
builder.add_edge(START, "parse_diff")

# parse_diff 후 3개 에이전트 병렬 실행
builder.add_edge("parse_diff", "security_agent")
builder.add_edge("parse_diff", "smell_agent")
builder.add_edge("parse_diff", "convention_agent")

# 3개 에이전트 결과를 merge_results로 통합
builder.add_edge("security_agent", "merge_results")
builder.add_edge("smell_agent", "merge_results")
builder.add_edge("convention_agent", "merge_results")

builder.add_edge("merge_results", END)

# 그래프 컴파일
review_graph = builder.compile()

print("LangGraph 파이프라인 컴파일 완료")
print("  START -> parse_diff -> [security | smell | convention] -> merge_results -> END")

In [ ]:
# --- 통합 파이프라인 실행 ---

import time

print("멀티에이전트 코드 리뷰 파이프라인 실행 시작...\n")
start_time = time.time()

result = review_graph.invoke({
    "pr_diff": SAMPLE_PR_DIFF,
    "file_diffs": [],
    "security_issues": [],
    "smell_issues": [],
    "convention_issues": [],
    "review_report": "",
})

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"파이프라인 완료! (소요 시간: {elapsed:.1f}초)")
print(f"{'='*60}")
print(f"\n보안 이슈: {len(result['security_issues'])}개")
print(f"코드 스멜: {len(result['smell_issues'])}개")
print(f"컨벤션 위반: {len(result['convention_issues'])}개")
print(f"\n{result['review_report']}")

---

## Section 9: 모델 라우팅 적용

EP16에서 배운 LLM FinOps 전략을 적용합니다.  
파일 크기와 유형에 따라 Sonnet(고성능) / Haiku(저비용) 모델을 선택합니다.

In [ ]:
# 보안 관련 키워드
SECURITY_KEYWORDS = ["auth", "security", "crypto", "login", "password", "token", "secret"]


def select_model_for_file(file_diff: FileDiff) -> str:
    """파일 특성에 따라 최적의 모델을 선택한다.
    
    라우팅 규칙:
    - 보안 관련 파일 -> 항상 Sonnet (안전 우선)
    - 100줄 이상 변경 -> Sonnet (복잡도 높음)
    - 50줄 미만 변경 -> Haiku (단순 변경)
    - 그 외 -> Sonnet (기본값)
    
    Returns:
        모델 이름 문자열
    """
    file_name = file_diff.file_path.lower()
    
    # 규칙 1: 보안 관련 파일
    if any(kw in file_name for kw in SECURITY_KEYWORDS):
        reason = f"보안 관련 키워드 감지 ({file_name})"
        model = "claude-sonnet-4-20250514"
    # 규칙 2: 대규모 변경
    elif file_diff.added_lines > 100:
        reason = f"대규모 변경 (+{file_diff.added_lines} lines)"
        model = "claude-sonnet-4-20250514"
    # 규칙 3: 소규모 변경
    elif file_diff.added_lines < 50:
        reason = f"소규모 변경 (+{file_diff.added_lines} lines)"
        model = "claude-haiku-4-5-20251001"
    # 기본값
    else:
        reason = "기본 라우팅"
        model = "claude-sonnet-4-20250514"
    
    print(f"  [routing] {file_diff.file_path} -> {model.split('-')[1]} ({reason})")
    return model


# 라우팅 테스트
print("=== 모델 라우팅 결과 ===")
for fd in file_diffs:
    selected = select_model_for_file(fd)

In [ ]:
# --- 비용 비교 시뮬레이션 ---

# 모델별 가격 (per 1M tokens, 2025년 기준 근사치)
PRICING = {
    "claude-sonnet-4-20250514": {"input": 3.0, "output": 15.0},
    "claude-haiku-4-5-20251001": {"input": 0.8, "output": 4.0},
}

def estimate_cost(file_diffs: list[FileDiff], tokens_per_line: int = 15) -> dict:
    """파일별 라우팅을 적용한 비용 추정."""
    sonnet_only_cost = 0.0
    routed_cost = 0.0
    
    for fd in file_diffs:
        input_tokens = fd.added_lines * tokens_per_line
        output_tokens = input_tokens * 2  # 리뷰 결과 추정
        
        # Sonnet 전용
        sonnet_price = PRICING["claude-sonnet-4-20250514"]
        sonnet_only_cost += (input_tokens * sonnet_price["input"] + output_tokens * sonnet_price["output"]) / 1_000_000
        
        # 라우팅 적용
        selected_model = select_model_for_file(fd)
        model_price = PRICING[selected_model]
        routed_cost += (input_tokens * model_price["input"] + output_tokens * model_price["output"]) / 1_000_000
    
    return {
        "sonnet_only": sonnet_only_cost,
        "routed": routed_cost,
        "savings_pct": (1 - routed_cost / sonnet_only_cost) * 100 if sonnet_only_cost > 0 else 0,
    }


print("\n=== 비용 비교 ===")
costs = estimate_cost(file_diffs)
print(f"\nSonnet 전용:   ${costs['sonnet_only']:.6f}")
print(f"라우팅 적용:   ${costs['routed']:.6f}")
print(f"절감률:        {costs['savings_pct']:.1f}%")
print("\n참고: 실제 비용은 프롬프트 크기, 컨텍스트 등에 따라 달라집니다.")

---

## Section 10: 품질 평가 (LLM-as-Judge)

EP05에서 배운 LLM-as-Judge 방식으로 AI 리뷰의 품질을 0~10점으로 평가합니다.

In [ ]:
class ReviewQualityScore(BaseModel):
    """리뷰 품질 평가 점수"""
    accuracy: int = Field(ge=0, le=10, description="정확성: 실제 문제를 정확히 짚었는가")
    specificity: int = Field(ge=0, le=10, description="구체성: 수정 방법이 구체적인가")
    priority: int = Field(ge=0, le=10, description="우선순위: 심각도 분류가 적절한가")
    readability: int = Field(ge=0, le=10, description="가독성: 리뷰 코멘트가 이해하기 쉬운가")
    overall: int = Field(ge=0, le=10, description="종합 점수")
    reasoning: str = Field(description="평가 근거 (한국어)")


EVALUATION_PROMPT = """당신은 코드 리뷰 품질 평가 전문가입니다.
아래에 PR diff와 AI가 생성한 리뷰 결과가 제공됩니다.
다음 기준으로 리뷰 품질을 0~10점으로 평가하세요:

1. accuracy (정확성): AI가 실제 문제를 정확히 짚었는가?
2. specificity (구체성): 수정 방법이 구체적이고 실행 가능한가?
3. priority (우선순위): 심각도 분류가 적절한가? (CRITICAL/HIGH가 실제로 심각한 문제인가?)
4. readability (가독성): 리뷰 코멘트가 이해하기 쉬운가?
5. overall (종합): 전체적인 리뷰 품질

reasoning에 평가 근거를 한국어로 작성하세요."""

print("리뷰 품질 평가 스키마 정의 완료")

In [ ]:
def evaluate_review_quality(pr_diff: str, review_report: str) -> ReviewQualityScore:
    """LLM-as-Judge로 AI 리뷰의 품질을 평가한다."""
    llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)
    structured_llm = llm.with_structured_output(ReviewQualityScore)
    
    user_msg = f"""## PR Diff
{pr_diff}

## AI 리뷰 결과
{review_report}

위 리뷰의 품질을 평가해주세요."""
    
    result = structured_llm.invoke([
        SystemMessage(content=EVALUATION_PROMPT),
        HumanMessage(content=user_msg),
    ])
    
    return result


# 품질 평가 실행
print("리뷰 품질 평가 중...\n")

quality = evaluate_review_quality(SAMPLE_PR_DIFF, result["review_report"])

print("=== 리뷰 품질 평가 결과 ===")
print(f"  정확성 (accuracy):    {quality.accuracy}/10")
print(f"  구체성 (specificity): {quality.specificity}/10")
print(f"  우선순위 (priority):  {quality.priority}/10")
print(f"  가독성 (readability): {quality.readability}/10")
print(f"  종합 (overall):       {quality.overall}/10")
print(f"\n평가 근거:\n{quality.reasoning}")

---

## Section 11: Langfuse 통합

Langfuse로 리뷰 실행을 트레이싱하고 품질 점수를 기록합니다.  
Langfuse 키가 설정되지 않은 경우 이 섹션은 건너뜁니다.

In [ ]:
if LANGFUSE_ENABLED:
    # Langfuse 콜백 핸들러 생성
    langfuse_handler = LangfuseCallbackHandler(
        tags=["ep21", "code-review-agent"],
        session_id="ep21-code-review-demo",
        metadata={"pr_files": [fd.file_path for fd in file_diffs]},
    )
    
    print("Langfuse 핸들러 생성 완료")
    print(f"  tags: ['ep21', 'code-review-agent']")
    print(f"  session_id: 'ep21-code-review-demo'")
else:
    print("Langfuse 미설정 - 이 셀을 건너뜁니다.")
    print("설정 방법: .env 파일에 LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY 추가")

In [ ]:
if LANGFUSE_ENABLED:
    # Langfuse 콜백과 함께 파이프라인 재실행
    print("Langfuse 트레이싱과 함께 파이프라인 실행...\n")
    
    traced_result = review_graph.invoke(
        {
            "pr_diff": SAMPLE_PR_DIFF,
            "file_diffs": [],
            "security_issues": [],
            "smell_issues": [],
            "convention_issues": [],
            "review_report": "",
        },
        config={"callbacks": [langfuse_handler]},
    )
    
    # 품질 점수 기록
    langfuse_client.score(
        trace_id=langfuse_handler.get_trace_id(),
        name="review_quality_overall",
        value=quality.overall,
        comment=f"정확성:{quality.accuracy} 구체성:{quality.specificity} 우선순위:{quality.priority} 가독성:{quality.readability}",
    )
    
    # 에이전트별 이슈 수 기록
    langfuse_client.score(
        trace_id=langfuse_handler.get_trace_id(),
        name="security_issues_count",
        value=len(traced_result["security_issues"]),
    )
    langfuse_client.score(
        trace_id=langfuse_handler.get_trace_id(),
        name="smell_issues_count",
        value=len(traced_result["smell_issues"]),
    )
    langfuse_client.score(
        trace_id=langfuse_handler.get_trace_id(),
        name="convention_issues_count",
        value=len(traced_result["convention_issues"]),
    )
    
    langfuse_client.flush()
    
    print("\nLangfuse 트레이싱 완료!")
    print(f"  Trace ID: {langfuse_handler.get_trace_id()}")
    print(f"  기록된 점수: review_quality_overall = {quality.overall}")
    print("  Langfuse 대시보드에서 트레이스 트리를 확인하세요.")
else:
    print("Langfuse 미설정 - 이 셀을 건너뜁니다.")

---

## Section 12: Exercises

아래 두 개의 Exercise를 직접 구현해보세요.

### Exercise 1: 성능 분석 에이전트 추가

기존 3개 에이전트(보안, 스멜, 컨벤션)에 **성능 분석 에이전트**를 추가하세요.

**요구사항:**
1. `PerformanceIssue` Pydantic 모델 정의 (type, severity, location, description, optimization)
2. 탐지 대상: N+1 쿼리 패턴, 불필요한 루프, 메모리 비효율, 블로킹 I/O
3. `performance_agent_node` 함수 구현
4. LangGraph 파이프라인에 4번째 에이전트 추가
5. `merge_results_node`에 성능 이슈 통합

In [ ]:
# Exercise 1: 성능 분석 에이전트 추가

# TODO 1: PerformanceIssue Pydantic 모델 정의
# class PerformanceIssue(BaseModel):
#     type: str = ...
#     severity: Literal["HIGH", "MEDIUM", "LOW"] = ...
#     location: str = ...
#     description: str = ...
#     optimization: str = ...


# TODO 2: 시스템 프롬프트 작성
# PERFORMANCE_SYSTEM_PROMPT = """..."""


# TODO 3: run_performance_agent 함수 구현
# def run_performance_agent(diff_content: str, model_name: str = "claude-sonnet-4-20250514"):
#     ...


# TODO 4: performance_agent_node 구현
# def performance_agent_node(state: ReviewState) -> dict:
#     ...


# TODO 5: LangGraph에 4번째 에이전트 추가 및 실행
# builder_v2 = StateGraph(ReviewState)
# ...


print("Exercise 1: TODO를 구현하세요!")

### Exercise 2: 리뷰 품질 자동 평가 시스템

여러 PR 샘플에 대해 리뷰를 실행하고, 리뷰 품질을 자동 평가하는 시스템을 구축하세요.

**요구사항:**
1. 추가 샘플 PR diff 2~3개 작성 (다양한 문제 포함)
2. 각 PR에 대해 리뷰 파이프라인 실행
3. LLM-as-Judge로 각 리뷰 품질 평가
4. 에이전트별 평균 점수 분석
5. (선택) Langfuse에 모든 평가 점수 기록

In [ ]:
# Exercise 2: 리뷰 품질 자동 평가 시스템

# TODO 1: 추가 샘플 PR diff 작성
# SAMPLE_PR_DIFF_2 = r"""
# diff --git a/app/api.py b/app/api.py
# ...
# """

# SAMPLE_PR_DIFF_3 = r"""
# diff --git a/app/models.py b/app/models.py
# ...
# """


# TODO 2: 여러 PR에 대해 리뷰 + 평가 실행
# all_diffs = [SAMPLE_PR_DIFF, SAMPLE_PR_DIFF_2, SAMPLE_PR_DIFF_3]
# results = []
# for i, diff in enumerate(all_diffs):
#     review = review_graph.invoke({...})
#     quality = evaluate_review_quality(diff, review["review_report"])
#     results.append({"pr": i+1, "review": review, "quality": quality})


# TODO 3: 에이전트별 평균 점수 분석
# for r in results:
#     print(f"PR #{r['pr']}: overall={r['quality'].overall}")


# TODO 4: (선택) Langfuse에 평가 점수 기록
# if LANGFUSE_ENABLED:
#     for r in results:
#         langfuse_client.score(...)


print("Exercise 2: TODO를 구현하세요!")

---

## 마무리

### 오늘 배운 것

1. **PR diff 파싱**: Unified diff format을 파일별로 분리하고 구조화하는 방법
2. **전문 분석 에이전트**: 보안 취약점, 코드 스멜, 컨벤션 체크를 각각 전문화된 에이전트로 구현
3. **멀티에이전트 파이프라인**: LangGraph `StateGraph`로 3개 에이전트 병렬 실행 + 결과 통합
4. **모델 라우팅**: 파일 크기/유형에 따라 Sonnet/Haiku 선택하여 비용 절감
5. **품질 평가**: LLM-as-Judge로 AI 리뷰 품질을 자동 평가
6. **Langfuse 추적**: 리뷰 실행 트레이싱 및 품질 점수 기록

### 활용한 이전 에피소드

| 에피소드 | 활용 내용 |
|----------|----------|
| EP05 벤치마크 | LLM-as-Judge 품질 평가 |
| EP10 멀티에이전트 | LangGraph 병렬 파이프라인 |
| EP12 MCP | GitHub MCP 서버 연동 (확장) |
| EP16 LLM FinOps | 파일별 모델 라우팅 |

### 다음 에피소드

**EP22. AI 미팅 에이전트** - 회의록 자동 생성과 액션 아이템 추출

---

> 전체 코드는 GitHub 레포에서, 심화 내용은 커뮤니티에서